[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/16_hyperparameter_optimization/01_grid_vs_random_search.ipynb)

# Bài 01 — Grid Search vs Random Search

**🎯 Mục tiêu**
- Hiểu bài toán tối ưu hyperparameter: hàm mục tiêu **đắt**, **nhiễu**, không có gradient.
- Vì sao **random search** thường tốt hơn grid search với cùng ngân sách (Bergstra & Bengio, 2012).
- Chọn **phân phối** tìm kiếm đúng: log-uniform cho learning rate / regularization, số nguyên cho số lá...
- Dùng `GridSearchCV`, `RandomizedSearchCV` với `scipy.stats`; đọc `cv_results_`.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/16_hyperparameter_optimization

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import loguniform, randint, uniform
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold

from ml_course.datasets import make_customer_churn
from ml_course.visualization import set_style

set_style()
churn = make_customer_churn(n=3000, seed=8)
X = pd.get_dummies(churn.drop(columns=["customer_id", "churn"]), dtype=float)
y = churn["churn"].to_numpy()
cv = StratifiedKFold(3, shuffle=True, random_state=0)

## 1. Bài toán

$$\lambda^* = \arg\max_{\lambda \in \Lambda}\ \text{CV-score}(\lambda)$$

- Mỗi lần đánh giá = train $k$ model → **đắt** (giây → giờ).
- Điểm CV **nhiễu** (phụ thuộc cách chia).
- Không có gradient theo $\lambda$; không gian hỗn hợp (liên tục, số nguyên, phân loại, điều kiện).
→ Cần chiến lược dùng **ít lần đánh giá** nhất.

## 2. Vì sao random search hiệu quả hơn?

Thường chỉ **vài** hyperparameter thực sự quan trọng (*low effective dimensionality*). Với ngân sách 9 lần thử trên 2 tham số:
- Grid 3 × 3: chỉ thử **3 giá trị khác nhau** của tham số quan trọng.
- Random: thử **9 giá trị khác nhau** của mỗi tham số.

In [ ]:
def objective(a, b):                    # chỉ a quan trọng (đỉnh hẹp quanh a = 0.62); b gần như vô ích
    return np.exp(-((a - 0.62) ** 2) / 0.005) + 0.05 * b


rng = np.random.default_rng(3)
grid_pts = np.array([(a, b) for a in [1 / 6, 0.5, 5 / 6] for b in [1 / 6, 0.5, 5 / 6]])
rand_pts = rng.uniform(0, 1, (9, 2))
aa = np.linspace(0, 1, 300)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, pts, name in zip(axes, [grid_pts, rand_pts], ["Grid search 3×3", "Random search 9 điểm"]):
    A, B = np.meshgrid(aa, aa)
    ax.contourf(A, B, objective(A, B), levels=20, cmap="viridis", alpha=0.7)
    ax.scatter(pts[:, 0], pts[:, 1], color="red", s=60, zorder=3)
    best = objective(pts[:, 0], pts[:, 1]).max()
    ax.set_title(f"{name}: tốt nhất = {best:.3f} | {len(np.unique(pts[:, 0].round(6)))} giá trị khác nhau của a")
    ax.set_xlabel("a (quan trọng)")
    ax.set_ylabel("b (không quan trọng)")
plt.show()

Lặp lại thí nghiệm nhiều lần để thấy đây không phải may rủi:

In [ ]:
def best_found(n_trials, method, seed):
    r = np.random.default_rng(seed)
    if method == "grid":
        side = int(np.sqrt(n_trials))
        offset = r.uniform(0, 1 / side)                      # dịch lưới ngẫu nhiên cho công bằng
        vals = (np.arange(side) / side + offset) % 1
        pts = np.array([(a, b) for a in vals for b in vals])
    else:
        pts = r.uniform(0, 1, (n_trials, 2))
    return objective(pts[:, 0], pts[:, 1]).max()


for n in [9, 16, 25]:
    g = [best_found(n, "grid", s) for s in range(500)]
    r_ = [best_found(n, "random", s) for s in range(500)]
    print(f"{n:2d} lần thử: grid tìm được đỉnh (> 0.9) {np.mean(np.array(g) > 0.9):.0%} số lần | random {np.mean(np.array(r_) > 0.9):.0%}")

## 3. Phân phối tìm kiếm

| Hyperparameter | Phân phối hợp lý | Vì sao |
|---|---|---|
| learning rate, `C`, `alpha`, `l2_regularization` | **log-uniform** (vd $10^{-4}$ → $10^{0}$) | tác động theo **bậc độ lớn**: 0.001 vs 0.01 khác nhau như 0.1 vs 1 |
| số lá, độ sâu, `n_neighbors` | số nguyên đều (hoặc log nếu khoảng rộng) | rời rạc |
| `subsample`, `max_features` (tỉ lệ) | uniform trên [0.5, 1] | tác động tuyến tính |
| kernel, loss, optimizer | phân loại (categorical) | |
| số cây / epoch | **không tune** — dùng early stopping | |

In [ ]:
samples = loguniform(1e-4, 1).rvs(10_000, random_state=0)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(uniform(1e-4, 1).rvs(10_000, random_state=0), bins=50)
axes[0].set_title("uniform(0, 1): 90% mẫu > 0.1 — bỏ phí vùng nhỏ")
axes[1].hist(np.log10(samples), bins=50)
axes[1].set_title("loguniform(1e-4, 1): đều theo log10 — mỗi bậc độ lớn như nhau")
axes[1].set_xlabel("log10(giá trị)")
plt.show()

## 4. Thực nghiệm: HistGradientBoosting trên churn

Cùng ngân sách **24 cấu hình** (× 3 fold):

In [ ]:
base = HistGradientBoostingClassifier(max_iter=100, early_stopping=False, random_state=0)
grid_space = {"learning_rate": [0.01, 0.05, 0.2, 0.5], "max_leaf_nodes": [4, 16, 64], "min_samples_leaf": [5, 50]}    # 4 × 3 × 2 = 24
random_space = {"learning_rate": loguniform(0.005, 0.5), "max_leaf_nodes": randint(2, 64), "min_samples_leaf": randint(5, 200)}

start = time.perf_counter()
gs = GridSearchCV(base, grid_space, cv=cv, scoring="roc_auc").fit(X, y)
t_grid = time.perf_counter() - start
start = time.perf_counter()
rs = RandomizedSearchCV(base, random_space, n_iter=24, cv=cv, scoring="roc_auc", random_state=0).fit(X, y)
t_rand = time.perf_counter() - start
print(f"Grid   ({len(gs.cv_results_['params'])} cấu hình, {t_grid:.0f}s): best ROC-AUC = {gs.best_score_:.4f} | {gs.best_params_}")
print(f"Random (24 cấu hình, {t_rand:.0f}s): best ROC-AUC = {rs.best_score_:.4f} | "
      + str({k: round(v, 4) if isinstance(v, float) else v for k, v in rs.best_params_.items()}))

In [ ]:
res = pd.DataFrame(rs.cv_results_)
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for ax, p in zip(axes, ["param_learning_rate", "param_max_leaf_nodes", "param_min_samples_leaf"]):
    ax.scatter(res[p].astype(float), res["mean_test_score"])
    ax.set_xlabel(p.replace("param_", ""))
    ax.set_ylabel("ROC-AUC (CV)")
axes[0].set_xscale("log")
plt.suptitle("Random search: điểm CV theo từng hyperparameter — tham số nào quan trọng?", y=1.03)
plt.show()

Trên bài toán này hai cách cho kết quả **gần như bằng nhau** — vùng cấu hình tốt khá rộng (nhiều tổ hợp khác nhau cho ROC-AUC ≈ 0.80). Lợi thế của random search lớn nhất khi có **nhiều** tham số mà chỉ vài cái quan trọng, và khi vùng tốt **hẹp** (mục 2).

Đồ thị phân tán theo từng tham số cho biết tham số nào **nhạy** (đường xu hướng rõ) → thu hẹp không gian cho vòng tìm kiếm sau. Đây là cách làm phổ biến: **random search rộng → thu hẹp → tìm tinh**.

## 5. Mẹo thực hành

- Bắt đầu với **mặc định** của thư viện làm baseline; tuning thường chỉ cải thiện vài phần trăm.
- Dùng early stopping cho số vòng/epoch thay vì tune chúng.
- Log-uniform cho mọi tham số "tỉ lệ / regularization".
- Kiểm tra tham số tốt nhất có nằm ở **biên** không gian không → nếu có, mở rộng không gian.
- `n_jobs=-1` để chạy song song; cố định `random_state` để tái lập.
- Điểm `best_score_` là lạc quan (chương 15) — đánh giá cuối trên test hoặc nested CV.

## 🧠 Kiểm tra nhanh

1. Vì sao random search hiệu quả hơn grid search khi chỉ vài tham số quan trọng?
2. Tại sao nên lấy mẫu learning rate theo log-uniform?
3. Tham số tốt nhất nằm đúng ở biên lưới — bạn làm gì?
4. Có nên tune số cây của gradient boosting bằng grid search không?

## 📌 Tóm tắt

- Tối ưu hyperparameter: hàm đắt, nhiễu, không gradient.
- Random search > grid search với cùng ngân sách (low effective dimensionality).
- Chọn phân phối đúng (log-uniform), early stopping cho số vòng, tìm rộng → thu hẹp.

## 📚 Tài liệu tham khảo

- Bergstra & Bengio (2012) — *Random Search for Hyper-Parameter Optimization*
- scikit-learn User Guide — Tuning the hyper-parameters of an estimator
- Probst, Boulesteix & Bischl (2019) — *Tunability: Importance of Hyperparameters of Machine Learning Algorithms*